In [3]:
import pandas as pd

def convert_monthly_to_avg_and_save(input_csv, output_csv):
    # Load dataset
    df = pd.read_csv(input_csv)

    # Variables with 12 monthly columns
    monthly_variables = [
        "EVI", "GCI", "NDVI", "NDWI",
        "ppt (mm)", "tmax (degrees C)", 
        "tmean (degrees C)", "tmin (degrees C)",
        "ssm", "susm"
    ]

    for var in monthly_variables:
        monthly_cols = [f"{var}_{m:02d}" for m in range(1, 13)]
        existing_cols = [col for col in monthly_cols if col in df.columns]

        if len(existing_cols) == 12:
            # Compute average and round to 2 decimals
            df[f"{var}_AVG"] = df[existing_cols].mean(axis=1).round(2)

            # Drop original monthly columns
            df.drop(columns=existing_cols, inplace=True)
        else:
            print(f"Warning: Missing columns for {var}")

    # Save to new CSV
    df.to_csv(output_csv, index=False)

    print("Saved cleaned dataset to:", output_csv)


# ==============================
# ✅ HOW TO RUN
# ==============================

convert_monthly_to_avg_and_save(
    input_csv="USDA_Corn_2016_2022.csv",
    output_csv="corn_dataset_avg.csv"
)

Saved cleaned dataset to: corn_dataset_avg.csv


C:\Users\PC\AppData\Local\Temp\ipykernel_9792\2026452865.py:21: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{var}_AVG"] = df[existing_cols].mean(axis=1).round(2)
C:\Users\PC\AppData\Local\Temp\ipykernel_9792\2026452865.py:21: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{var}_AVG"] = df[existing_cols].mean(axis=1).round(2)
C:\Users\PC\AppData\Local\Temp\ipykernel_9792\2026452865.py:21: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor

In [3]:
import pandas as pd

df = pd.read_csv("corn_dataset_avg.csv")

# 1️⃣ Sort properly
df = df.sort_values(["state_name", "Latitude", "Longitude", "year"])

# 2️⃣ Compute historical average yield per county (state + lat + lon)
df["HIST_AVG_YIELD"] = (
    df.groupby(["state_name", "Latitude", "Longitude"])["YIELD"]
      .apply(lambda x: x.expanding().mean().shift(1))
      .reset_index(level=[0,1,2], drop=True)
)

# 3️⃣ Fill first-year NaNs with that year's YIELD
df["HIST_AVG_YIELD"] = df["HIST_AVG_YIELD"].fillna(df["YIELD"])

# 4️⃣ Round to 1 decimal place
df["HIST_AVG_YIELD"] = df["HIST_AVG_YIELD"].round(1)

# 5️⃣ Restore original row order (optional)
df = df.sort_index()

print(df.head(10))

     id state_name  year  YIELD   AWC   SOM    CEC  Latitude  Longitude  \
0  1005    ALABAMA  2016  188.9  0.12  0.84  13.63     31.86      -85.4   
1  1005    ALABAMA  2017  192.3  0.12  0.84  13.63     31.86      -85.4   
2  1005    ALABAMA  2019  184.1  0.12  0.84  13.63     31.86      -85.4   
3  1005    ALABAMA  2020  151.7  0.12  0.84  13.63     31.86      -85.4   
4  1005    ALABAMA  2021  188.8  0.12  0.84  13.63     31.86      -85.4   
5  1005    ALABAMA  2022  159.8  0.12  0.84  13.63     31.86      -85.4   
6  1019    ALABAMA  2016   92.3  0.14  0.92  14.76     34.11      -85.6   
7  1019    ALABAMA  2017  146.8  0.14  0.92  14.76     34.11      -85.6   
8  1019    ALABAMA  2019  129.4  0.14  0.92  14.76     34.11      -85.6   
9  1019    ALABAMA  2020  122.4  0.14  0.92  14.76     34.11      -85.6   

   EVI_AVG  GCI_AVG  NDVI_AVG  NDWI_AVG  ppt (mm)_AVG  tmax (degrees C)_AVG  \
0     0.38    10.90      0.70      0.50         94.41                 26.26   
1     0.38    10

In [1]:
import pandas as pd

df = pd.read_csv("corn_dataset_avg.csv")

# 1️⃣ Sort by state and year
df = df.sort_values(["state_name", "year"])

# 2️⃣ Compute historical average yield (previous years only)
df["HIST_AVG_YIELD"] = (
    df.groupby("state_name")["YIELD"]
      .apply(lambda x: x.expanding().mean().shift(1))
      .reset_index(level=0, drop=True)
)

# 3️⃣ Fill NaN values with actual YIELD
df["HIST_AVG_YIELD"] = df["HIST_AVG_YIELD"].fillna(df["YIELD"])

# 4️⃣ Round to 1 decimal place
df["HIST_AVG_YIELD"] = df["HIST_AVG_YIELD"].round(1)

# 5️⃣ Restore original order (optional)
df = df.sort_index()

print(df.head(10))


     id state_name  year  YIELD   AWC   SOM    CEC  Latitude  Longitude  \
0  1005    ALABAMA  2016  188.9  0.12  0.84  13.63     31.86      -85.4   
1  1005    ALABAMA  2017  192.3  0.12  0.84  13.63     31.86      -85.4   
2  1005    ALABAMA  2019  184.1  0.12  0.84  13.63     31.86      -85.4   
3  1005    ALABAMA  2020  151.7  0.12  0.84  13.63     31.86      -85.4   
4  1005    ALABAMA  2021  188.8  0.12  0.84  13.63     31.86      -85.4   
5  1005    ALABAMA  2022  159.8  0.12  0.84  13.63     31.86      -85.4   
6  1019    ALABAMA  2016   92.3  0.14  0.92  14.76     34.11      -85.6   
7  1019    ALABAMA  2017  146.8  0.14  0.92  14.76     34.11      -85.6   
8  1019    ALABAMA  2019  129.4  0.14  0.92  14.76     34.11      -85.6   
9  1019    ALABAMA  2020  122.4  0.14  0.92  14.76     34.11      -85.6   

   EVI_AVG  GCI_AVG  NDVI_AVG  NDWI_AVG  ppt (mm)_AVG  tmax (degrees C)_AVG  \
0     0.38    10.90      0.70      0.50         94.41                 26.26   
1     0.38    10

In [ ]:
df.to_csv("corn_dataset_final.csv", index=False)

In [15]:
import pandas as pd

# Load your dataset
df = pd.read_csv("corn_dataset_final.csv")

# Get unique combinations
unique_combos = df[['state_name']].drop_duplicates()

print(len(unique_combos))


34
